<a href="https://colab.research.google.com/github/SayanB58/HAAI__Cohort2_LLM_Tokenizers_Embeddings/blob/main/HAAI_Cohort2_Week4_AgenticAI_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This program is build with Flan-T5-XL LLM to be able to answer the final question using the output from the previous questions as in-context learning/few-shot learning.

Consider three related questions from a search session: Question 1, Question 2, Question 3
1. Answer to Question 1 needs to be generated.
2. Answer to Question 2 needs to be generated with the answer to Question 1 as one-shot example / context.
3. Answer to Question 3 needs to be generated with the answer to Question 2 as one-shot example / context.
4. Answer to Question 3 will be either YES or NO and nothing else.


* The program accepts three parameters provided as a command line input.
* The three inputs represent the questions.
* The output of the first two question is Generation based whereas the last question output is deterministic i.e. its either YES or NO.
* The output of the first two question is Generation based whereas the last question output is deterministic i.e. its either YES or NO.
* Output should be in upper-case: YES or NO
* There should be no additional output including any warning messages in the terminal.
* Remember that your output will be tested against test cases, therefore any deviation from the test cases will be considered incorrect during evaluation.

Syntax: python template.py `<string>` `<string>` `<string>`

The following example is given for your reference:

 Terminal Input: python template.py "Who is Rabindranath Tagore?" "Where was he born?" "Is it in America?"
Terminal Output: NO

 Terminal Input: python template.py "Who is Rabindranath Tagore?" "Where was he born?" "Is it in India?"
Terminal Output: YES

You are expected to create some examples of your own to test the correctness of your approach.

ALL THE BEST!!

ALERT: **No changes are allowed to import statements**

In [2]:
import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

##### You may comment this section to see verbose -- but you must un-comment this before final submission. ######
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
#################################################################################################################

In [3]:
from google.colab import userdata
#userdata.get('HF_TOKEN')

**Changes allowed from here**

In [4]:
def llm_function(model,tokenizer,questions):
    '''
    The steps are given for your reference:

    1. Generate answer for the first question.
    2. Generate answer for the second question use the answer for first question as context.
    3. Generate a deterministic output either 'YES' or 'NO' for the third question using the context from second question.
    5. Clean output and return.
    6. Output is case-sensative: YES or NO
    Note: The model (Flan-T5-XL) and tokenizer is already initialized. Do not modify that section.
    '''
    q1, q2, q3 = questions[0], questions[1], questions[2]

    # Helper to generate deterministic text from the model
    def generate_text(prompt, max_length=128):
        tokenized_inputs = tokenizer(prompt, return_tensors="pt")
        #print(tokenized_inputs)
        inputs = tokenized_inputs.input_ids
        #print(inputs)
        attention_mask = tokenized_inputs.attention_mask
        # Use beam search (deterministic) and no sampling
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )
        #print(outputs)
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        #print(text)
        return text.strip()

    # 1) Answer Question 1
    prompt1 = f"Question: {q1}\nAnswer:"
    ans1 = generate_text(prompt1, max_length=128)

    # 2) Answer Question 2 using Q1/ans1 as one-shot context
    prompt2 = (
        f"Question: {q1}\nAnswer: {ans1}\n\n"
        f"Question: {q2}\nAnswer:"
    )
    ans2 = generate_text(prompt2, max_length=128)

    # 3) Answer Question 3 using Q2/ans2 as one-shot context and force YES/NO
    # We instruct the model to respond with only YES or NO.
    prompt3 = (
        f"Question: {q2}\nAnswer: {ans2}\n\n"
        f"Question: {q3}\nAnswer with only YES or NO (no other text):"
    )
    ans3_raw = generate_text(prompt3, max_length=8)

    # Normalize model output to a strict YES or NO
    ans3 = ans3_raw.strip().upper()
    # Extract first occurrence of YES or NO if present
    if re.search(r"\bYES\b", ans3):
        final_output = "YES"
    elif re.search(r"\bNO\b", ans3):
        final_output = "NO"
    else:
        # Fallback: try to interpret common affirmative/negative words
        low = ans3_raw.strip().lower()
        if low.startswith("y") or "yes" in low or "true" in low or "correct" in low:
            final_output = "YES"
        else:
            final_output = "NO"

    return final_output

ALERT: **No changes are allowed below this comment**

In [5]:
if __name__ == '__main__':

    # question_a = sys.argv[1].strip()
    # question_b = sys.argv[2].strip()
    # question_c = sys.argv[3].strip()

    question_a = "Who is Rabindranath Tagore?"
    question_b = "Where was he born?"
    question_c = "Is it in India?"

    questions = [question_a, question_b, question_c]
    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    out = llm_function(model,tokenizer,questions)
    print(out.strip())

    """ End to call """

NO
